[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 困难：KV 缓存注意力机制

实现带有 **KV 缓存的多头注意力机制**，用于高效的自回归生成。

在 LLM 推理过程中，每一步都重新计算所有键/值投影非常浪费资源。
**KV 缓存**将之前计算的 K 和 V 张量保存下来，这样只需对新 token 进行投影。

### 函数签名
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — 新 token
        # cache: None 或 (K_past, V_past)，每个形状为 (B, num_heads, S_past, d_k)
        # 返回: (output, (K_all, V_all))
```

### 要求
- 继承自 `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`：`nn.Linear` 投影层
- 当 `cache=None`（预填充阶段）：应用**因果掩码**，将所有 K/V 作为缓存返回
- 当提供 `cache`（解码阶段）：将新 K/V 与缓存拼接，单 token 解码无需因果掩码
- 增量解码的结果必须与完整前向传播**完全一致**

### 核心思路
```
预填充:  [t0 t1 t2 t3] → 完整因果注意力 → cache = (K_{0:3}, V_{0:3})
解码:    [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
解码:    [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

import torch.nn.functional as F

In [ ]:
# ✏️ 在此实现你的代码

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # 初始化 W_q, W_k, W_v, W_o

    def forward(self, x, cache=None):
        # 1. 从 x 投影得到 Q, K, V
        # 2. 重塑为多头形式: (B, num_heads, S, d_k)
        # 3. 若缓存存在，将新 K/V 与缓存 K/V 拼接
        # 4. 计算注意力（预填充阶段需要因果掩码）
        # 5. 返回 (output, (K_all, V_all))
        pass

kv_cache 是在 k 和 v 上缓存 seq_len 维度, k(v) 是 k(v)_past + k(v)_new 拼接，然后 q * k_T 得到 (b, h, q_new_seq_len, full_seq_len) 和 (b, h, full_seq_len, dim) 进行点乘

In [ ]:
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D)
        batch_size, seq_len, _ = x.shape
        
        # 1. 投影并重塑为多头形式: (B, num_heads, S_new, d_k)
        q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 2. KV 缓存处理
        if cache is not None:
            k_past, v_past = cache
            # 拼接历史缓存与当前新计算的 K, V, 注意拼接维度是 seq_len
            # 形状变化: (B, num_heads, S_past + S_new, d_k)
            k = torch.cat([k_past, k], dim=2)
            v = torch.cat([v_past, v], dim=2)
        
        # 当前全量 K, V 将作为新的缓存返回
        new_cache = (k, v)
        
        # 3. 计算注意力分数
        # q: (B, H, S_new, d_k), k: (B, H, S_all, d_k)
        # scores: (B, H, S_new, S_all)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 4. 掩码处理 (Masking)
        # 预填充阶段 (S_new > 1) 需要因果掩码；
        # 增量解码阶段 (S_new == 1) 通常不需要，因为 Q 只有一个 token，它天然只能看到之前的 K
        if seq_len > 1:
            # 创建下三角矩阵掩码
            # 注意：掩码维度必须匹配 (S_new, S_all)
            s_all = k.size(2)
            mask = torch.tril(torch.ones(seq_len, s_all, device=x.device))
            # 如果是增量步，需要偏移掩码以确保对齐
            if cache is not None:
                # 实际上在解码阶段 seq_len=1，此逻辑不会触发，
                # 但为了健壮性，处理 S_new > 1 且有 cache 的情况：
                mask = mask[:, -seq_len:] # 确保 mask 长宽一致
            
            scores = scores.masked_fill(mask == 0, float('-inf')) # K 位置大于等于 Q 的 seq_len 位置都置 -∞

        # 5. Softmax + 加权求和
        attn = F.softmax(scores, dim=-1)
        # context: (B, H, S_new, d_k)
        context = torch.matmul(attn, v)

        # 6. 还原形状并输出投影
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(context)

        return output, new_cache

In [ ]:
# 🧪 调试
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# 完整前向传播
full_out, _ = attn(x)
print("完整输出形状:", full_out.shape)  # (1, 6, 64)

# 增量解码: 预填充 4 个 token，再解码 1+1
out1, cache = attn(x[:, :4])
print("缓存 K 形状:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("结果一致:", torch.allclose(full_out, inc_out, atol=1e-5))

In [ ]:
# ✅ 提交
from torch_judge import check
check('kv_cache')